# Calabi-Yau metric groups — v4 on Colab

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Cells in order: 1) GPU check · 2) install (≈3 min) · 3) GPU binding test · 4) optional Drive · 5) write script · 6) run.

Colab's Python is too new for cymetric, so we build a Python 3.11 environment with `uv` and run the script inside it.

In [ ]:
!nvidia-smi -L

In [ ]:
%%bash
set -e
pip install -q uv
uv venv cy311 --python 3.11 -q
. cy311/bin/activate
uv pip install -q "git+https://github.com/pythoncymetric/cymetric.git" \
  "nvidia-cudnn-cu11==8.7.0.84" "nvidia-cublas-cu11==11.11.3.6" "nvidia-cuda-runtime-cu11==11.8.89" \
  "nvidia-cuda-nvrtc-cu11==11.8.89" "nvidia-cufft-cu11==10.9.0.58" "nvidia-curand-cu11==10.3.0.86" \
  "nvidia-cusolver-cu11==11.4.1.48" "nvidia-cusparse-cu11==11.7.5.86" "nvidia-cuda-cupti-cu11==11.8.87" "nvidia-nccl-cu11==2.16.5"
# runner that points TensorFlow at the pip-installed CUDA libraries
cat > run.sh <<'SH'
#!/bin/bash
. /content/cy311/bin/activate
export LD_LIBRARY_PATH=$(python -c "import os,nvidia;p=os.path.dirname(nvidia.__path__[0]) if hasattr(nvidia,'__path__') else '';import glob;print(':'.join(glob.glob('/content/cy311/lib/python3.11/site-packages/nvidia/*/lib')))" ):$LD_LIBRARY_PATH
export TF_CPP_MIN_LOG_LEVEL=2
exec python "$@"
SH
chmod +x run.sh
echo INSTALL DONE

In [ ]:
!./run.sh -c "import tensorflow as tf, cymetric; print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))"


If the line above says `GPUs: []`, tell Claude — we'll switch backend. If it lists a GPU, carry on.

In [ ]:
# OPTIONAL: keep sampled data + logs across sessions (Colab wipes /content when the runtime dies)
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/cy_v4', exist_ok=True)
%cd /content/drive/MyDrive/cy_v4

In [ ]:
%%writefile moduli_net_v4.py
# Stage-2 prototype v4: TWO groups, a fingerprint ROUTER, and proper training.
#   - Two base recipes A and B, each with a neighbourhood of perturbations.
#   - One specialist per group, trained with cymetric's two-phase scheme + LR decay.
#   - Fingerprint router: a new shape is sent to the nearest group centre.
#   - Test shapes: held-out neighbours of A, of B, and strangers.
#     Each is trained three ways: scratch / warm from ROUTED specialist / warm from the OTHER specialist.
#   Set QUICK=True for a laptop smoke test; QUICK=False for the real (GPU) run.
import os, sys, csv, itertools, time, numpy as np, tensorflow as tf
tf.get_logger().setLevel('ERROR')
tfk = tf.keras
from cymetric.pointgen.pointgen_cicy import CICYPointGenerator
from cymetric.models.tfhelper import prepare_tf_basis
from cymetric.models.tfmodels import PhiFSModel
from cymetric.models.metrics import TotalLoss, SigmaLoss

QUICK = ('--quick' in sys.argv)
CFG = dict(
    N_NEIGH   = 4  if QUICK else 12,     # training neighbours per group
    N_HELD    = 1  if QUICK else 3,      # held-out neighbours per group
    N_STRANGE = 1  if QUICK else 3,
    EPS       = 0.05,
    N_POINTS  = 2000 if QUICK else 50000,
    E_SPEC    = 4  if QUICK else 100,    # specialist epochs
    E_WARM    = 3  if QUICK else 30,     # fine-tune / scratch epochs
    HIDDEN    = 64 if QUICK else 256,
    LAYERS    = 3  if QUICK else 4,
    LR        = 1e-3,
    LR_WARM   = 3e-4,                    # gentler LR when fine-tuning from a specialist
)
globals().update(CFG)
DATA = 'moduli_data_v4'; LOG = 'moduli_v4_log.csv'
print("CONFIG:", CFG, "\nGPUs:", tf.config.list_physical_devices('GPU'))

def deg3(n): return [m for m in itertools.product(range(4), repeat=n) if sum(m)==3]
MONOMIALS = np.array([a+b for a in deg3(3) for b in deg3(3)], dtype=np.int64)
AMBIENT   = np.array([2, 2]); KMODULI = np.ones(2)

def recipe(seed, base=None):
    rng = np.random.default_rng(seed)
    c = rng.normal(size=100) + 1j*rng.normal(size=100); c /= np.linalg.norm(c)
    if base is not None: c = base + EPS*c; c /= np.linalg.norm(c)
    return c

def make_shape(name, coeffs):
    d = f'{DATA}/{name}'
    if not os.path.exists(os.path.join(d, 'basis.pickle')):
        pg = CICYPointGenerator([MONOMIALS], [coeffs], KMODULI, AMBIENT, verbose=0)
        kappa = pg.prepare_dataset(N_POINTS, d); pg.prepare_basis(d, kappa=kappa)
    data  = np.load(os.path.join(d, 'dataset.npz'))
    BASIS = prepare_tf_basis(np.load(os.path.join(d, 'basis.pickle'), allow_pickle=True))
    return dict(name=name, c=coeffs, d=data, B=BASIS)

def new_core():
    layers = [tfk.Input(shape=(12,))] + [tfk.layers.Dense(HIDDEN, activation='gelu') for _ in range(LAYERS)]
    return tfk.Sequential(layers + [tfk.layers.Dense(1, use_bias=False)])

def wrap(core, BASIS, lr):
    m = PhiFSModel(core, BASIS, alpha=[1.,1.,1.,1.,1.])
    m.compile(custom_metrics=[TotalLoss(), SigmaLoss()], optimizer=tfk.optimizers.legacy.Adam(lr))
    return m

def sigma(m, d): return float(m.evaluate(d['X_val'], d['y_val'], batch_size=2000, verbose=0)[1])

def two_phase_epoch(m, d):
    """cymetric's recipe: small batches w/o volume loss, then large batches with it."""
    m.learn_volk = tf.cast(False, tf.bool)
    m.fit(d['X_train'], d['y_train'], batch_size=64, epochs=1, verbose=0)
    m.learn_volk = tf.cast(True, tf.bool)
    m.fit(d['X_train'], d['y_train'], batch_size=min(10000, len(d['X_train'])), epochs=1, verbose=0)

def set_lr(models, lr):
    for m in models: m.optimizer.learning_rate.assign(lr)

def fingerprint(s):
    w = s['d']['y_train'][:, 0]; om = s['d']['y_train'][:, 1]
    zero = tfk.Sequential([tfk.Input(shape=(12,)), tfk.layers.Lambda(lambda x: 0*x[:, :1])])
    return np.array([sigma(wrap(zero, s['B'], 1e-3), s['d']), np.std(np.log(w)), np.std(np.log(om)), np.max(w)/np.mean(w)])

def train_specialist(group, tag):
    core = new_core(); ms = [wrap(core, s['B'], LR) for s in group]
    t0 = time.time()
    for ep in range(1, E_SPEC+1):
        set_lr(ms, LR * 0.5**(ep / (E_SPEC/3)))             # decay to ~LR/8 by the end
        for m, s in zip(ms, group): two_phase_epoch(m, s['d'])
        if ep in (1, 2, 5) or ep % 10 == 0 or ep == E_SPEC:
            print(f"   [{tag}] epoch {ep:3d}  mean sigma {np.mean([sigma(m, s['d']) for m, s in zip(ms, group)]):.4f}  ({time.time()-t0:.0f}s)", flush=True)
    core.save_weights(f'specialist_{tag}.weights.h5')
    return core.get_weights()

def finetune(s, init_w, lr, tag):
    k = new_core()
    if init_w is not None: k.set_weights(init_w)
    m = wrap(k, s['B'], lr); cur = [sigma(m, s['d'])]
    for ep in range(1, E_WARM+1):
        set_lr([m], lr * 0.5**(ep / (E_WARM/3)))
        two_phase_epoch(m, s['d']); cur.append(sigma(m, s['d']))
    return cur

def main():
    print("Sampling shapes (cached after first run)...", flush=True)
    baseA, baseB = recipe(0), recipe(1)
    grpA = [make_shape(f'A_{i}', recipe(100+i, baseA)) for i in range(N_NEIGH)]
    grpB = [make_shape(f'B_{i}', recipe(150+i, baseB)) for i in range(N_NEIGH)]
    tests = ([('heldA', make_shape(f'heldA_{i}', recipe(200+i, baseA))) for i in range(N_HELD)] +
             [('heldB', make_shape(f'heldB_{i}', recipe(250+i, baseB))) for i in range(N_HELD)] +
             [('stranger', make_shape(f'strange_{i}', recipe(300+i))) for i in range(N_STRANGE)])

    print("\nA. FINGERPRINTS + ROUTER", flush=True)
    for s in grpA + grpB + [t for _, t in tests]: s['fp'] = fingerprint(s)
    allfp = np.array([s['fp'] for s in grpA + grpB]); mu, sd = allfp.mean(0), allfp.std(0) + 1e-9
    z = lambda s: (s['fp'] - mu) / sd
    cA, cB = np.mean([z(s) for s in grpA], 0), np.mean([z(s) for s in grpB], 0)
    print(f"   group centres are {np.linalg.norm(cA-cB):.2f} apart (normalised units)")
    for kind, s in tests:
        dA, dB = np.linalg.norm(z(s)-cA), np.linalg.norm(z(s)-cB)
        s['route'] = 'A' if dA < dB else 'B'
        print(f"   {kind:9s} {s['name']:10s} dist A {dA:5.2f}  dist B {dB:5.2f}  -> routed to {s['route']}", flush=True)

    print(f"\nB. SPECIALISTS ({N_NEIGH} shapes each, {E_SPEC} epochs)", flush=True)
    wA = train_specialist(grpA, 'A'); wB = train_specialist(grpB, 'B')
    W = {'A': wA, 'B': wB}

    print(f"\nC. TEST SHAPES: scratch vs warm(routed) vs warm(other), {E_WARM} epochs", flush=True)
    rows = []
    for kind, s in tests:
        r, o = s['route'], ('B' if s['route'] == 'A' else 'A')
        curves = {'scratch': finetune(s, None, LR, 'scratch'),
                  f'warm-{r} (routed)': finetune(s, W[r], LR_WARM, 'routed'),
                  f'warm-{o} (other)':  finetune(s, W[o], LR_WARM, 'other')}
        for mode, cur in curves.items():
            rows.append([kind, s['name'], mode] + cur)
            print(f"   {kind:9s} {s['name']:10s} {mode:18s} start {cur[0]:.3f}  best {min(cur):.3f}  end {cur[-1]:.3f}", flush=True)
    with open(LOG, 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['group', 'shape', 'mode'] + [f'ep{i}' for i in range(E_WARM+1)]); w.writerows(rows)
    print(f"\nLog: {LOG}. Read: routed should beat other AND scratch for held-out shapes; strangers show how far the specialists generalise.")

if __name__ == '__main__':
    main()


First a quick smoke test (~3 min), then the real run. Sampling 50k points × 24 shapes takes a while even on GPU (point generation is CPU-bound); the training then flies.

In [ ]:
!/content/run.sh moduli_net_v4.py --quick

In [ ]:
!/content/run.sh moduli_net_v4.py

In [ ]:
# Plot the fine-tuning curves from the log
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv('moduli_v4_log.csv'); eps = [c for c in df.columns if c.startswith('ep')]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, grp in zip(axes, ['heldA', 'heldB', 'stranger']):
    for _, r in df[df.group == grp].iterrows():
        ax.plot(range(len(eps)), r[eps].values, label=f"{r['shape']} {r['mode']}", alpha=.8)
    ax.set_title(grp); ax.set_xlabel('epoch'); ax.legend(fontsize=6)
axes[0].set_ylabel('sigma (Ricci error)'); plt.tight_layout(); plt.show()